# Detailed PEN Explore 2 

### Dependencies / DIR / DB Connect

In [7]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
import sqlite3


# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

# one directory up (your config.py lives here)
config_folder = base_dir.parent

# two directories up (for TEMP, data, images)
project_root = base_dir.parent.parent



# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP" / "PEN"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scatter_plots"

# ======= IMPORT CONFIG =======
# ====== DB Path settings Ect ======
sys.path.insert(0, str(config_folder))
import config  # now you can import config.py

# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")
school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

conn = sqlite3.connect(data_folder /config.recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)
# Print table names to verify connection
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print(table[0])

Connected to database: ../data/db/Season_YTD.db
Tables in the database:
game_details
scoring_summary
penalty_summary
goalie_stats
player_stats
line_chart
linescore
advanced_metrics
shot_events
player_stats_ytd


### Build Model to get manpower for each team at any point based on penalty summary table

flowchart TD
  A[Penalty event at (Game,Period,Time)] --> B{Manpower-eligible?}
  B -- No --> B0[Box time only\n(No manpower change)] --> Z
  B -- Yes --> C[Group penalties by same timestamp]

  C --> D{Both teams have manpower-eligible\npenalties at this timestamp?}
  D -- Yes --> E[Pair off min(countA,countB)\n= coincidental pairs\n(mark is_coincidental=True)]
  E --> F[Unpaired penalties remain\n= non-coincidental]
  D -- No --> F

  F --> G[Start/queue rule:\nIf penalty would reduce team below 3 skaters,\nqueue it (doesn't affect manpower yet)\nElse it affects manpower now]
  G --> Z[Update game state timeline]

  Z --> H[Goal event]
  H --> I{Does scoring team have\nmore skaters than opponent\nat goal time?}
  I -- No --> J[No cancellation] --> Z
  I -- Yes --> K{Does defending team have\ncancelable minor affecting manpower?}
  K -- No --> J --> Z
  K -- Yes --> L[Cancel exactly ONE\nactive manpower minor\n(earliest-expiring)]
  L --> Z


In [8]:
pen_raw   = pd.read_sql("SELECT * FROM penalty_summary;", conn)
goals_raw = pd.read_sql("SELECT * FROM scoring_summary;", conn)

import pandas as pd
import numpy as np
from collections import deque, defaultdict

# ============================================================
# TRUE ENGINE (D1-style):
# - Coincidental pairing ONLY at exact same timestamp
# - Coincidental penalties DO reduce skaters (4v4 etc)
# - Penalty CLOCK QUEUEING: max 2 skater-reducing penalties per team "serving" at once
# - Goal cancellation: cancels ONE *non-coincidental* minor only, ONLY if strict advantage exists
# - Majors never cancel; goals at 4v4/3v3 don't cancel anything
# - Pre-goal state is what we compare to scoring "PP" notes
# ============================================================


OT_LENGTH = 300.0  # seconds of OT to simulate if OT appears

# --------------------
# Time helpers
# --------------------
def period_start_seconds(period: str) -> int:
    p = str(period).strip()
    return {
        "1st Period": 0,
        "2nd Period": 20 * 60,
        "3rd Period": 40 * 60,
        "Overtime":   60 * 60,
        "OT":         60 * 60,
    }.get(p, 0)

def parse_mmss(x) -> int:
    s = str(x).strip()
    if ":" not in s:
        try:
            return int(float(s))
        except Exception:
            return 0
    m, sec = s.split(":")
    return int(m) * 60 + int(sec)

def abs_time_seconds(df: pd.DataFrame, period_col="Period", time_col="Time") -> pd.Series:
    return df[time_col].apply(parse_mmss) + df[period_col].apply(period_start_seconds)

# --------------------
# Situation label (your spec)
# --------------------
def sit_label(team_skaters: int, opp_skaters: int, phase: str) -> str:
    if team_skaters == 3 and opp_skaters == 3:
        return "3v3 (OT)" if phase == "OT" else "3v3 (NON OT)"
    return f"{team_skaters}v{opp_skaters}"

# --------------------
# Which penalties remove a skater in this model?
# We use Pen_Length only (since Penalty_Type text varies).
# - 2: minor (cancelable)
# - 4: double minor -> split into 2 + 2 (cancelable halves)
# - 5: major (not cancelable)
# Everything else ignored here (misconducts etc)
# --------------------
def manpower_eligible_len(L) -> bool:
    try:
        return int(L) in (2, 4, 5)
    except Exception:
        return False

# --------------------
# Split double minors into two sequential 2-min "assessments"
# at t and t+120 (game clock), for pairing/queue purposes.
# --------------------
def split_double_minors(pen: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, r in pen.iterrows():
        L = int(r["Pen_Length"])
        if L == 4:
            r1 = r.copy()
            r2 = r.copy()
            r1["Pen_Length"] = 2
            r2["Pen_Length"] = 2
            r1["double_half"] = 1
            r2["double_half"] = 2
            r2["t_call"] = float(r["t_call"]) + 120.0
            # keep original Period/Time for audit; but call time shifts
            out.extend([r1, r2])
        else:
            rr = r.copy()
            rr["double_half"] = 0
            out.append(rr)
    return pd.DataFrame(out).reset_index(drop=True)

# --------------------
# Scoring PP-note mapping for mismatch checks
# (we only validate strength tokens, ignore EN/EA/etc beyond first token)
# --------------------
def expected_from_pp_note(pp_note):
    if pd.isna(pp_note):
        return "5v5"
    token = str(pp_note).split(",")[0].strip()
    return {
        "5x3": "5v3",
        "4x4": "4v4",
        "3x5": "3v5",
        "3x3": "3v3",   # accept either OT/NON OT in comparison logic
    }.get(token, None)

def pp_strength_token(pp_note):
    if pd.isna(pp_note):
        return None
    return str(pp_note).split(",")[0].strip()

# ============================================================
# PREP INPUTS
# ============================================================
pen = pen_raw.copy()
goals = goals_raw.copy()

# Normalize lengths and keep eligible
pen["Pen_Length"] = pd.to_numeric(pen["Pen_Length"], errors="coerce")
pen = pen[pen["Pen_Length"].notna()].copy()
pen["Pen_Length"] = pen["Pen_Length"].astype(int)
pen = pen[pen["Pen_Length"].apply(manpower_eligible_len)].copy()

# Absolute call times
pen["t_call"] = abs_time_seconds(pen, "Period", "Time").astype(float)
goals["t_goal"] = abs_time_seconds(goals, "Period", "Time").astype(float)

# Split doubles into two 2-min assessments
pen = split_double_minors(pen)

# Give stable IDs
pen = pen.reset_index(drop=True)
pen["Penalty_ID"] = pen.index.astype(int)

# Game teams
if {"Away_Team", "Home_Team"}.issubset(goals.columns):
    game_teams = goals.groupby("Game_ID")[["Away_Team", "Home_Team"]].first().reset_index()
else:
    tmp = goals.groupby("Game_ID")["Team"].unique().reset_index()
    rows = []
    for _, r in tmp.iterrows():
        teams = list(r["Team"])
        if len(teams) >= 2:
            rows.append({"Game_ID": r["Game_ID"], "Away_Team": teams[0], "Home_Team": teams[1]})
    game_teams = pd.DataFrame(rows)

# ============================================================
# PER-GAME EVENT SIMULATOR
# ============================================================
def simulate_game(game_id: str, away: str, home: str):
    pen_game = pen[pen["Game_ID"] == game_id].copy()
    goals_game = goals[goals["Game_ID"] == game_id].copy()

    # Determine end_time: regulation unless any event in OT
    max_t = 0.0
    if not pen_game.empty:
        max_t = max(max_t, float(pen_game["t_call"].max()))
    if not goals_game.empty:
        max_t = max(max_t, float(goals_game["t_goal"].max()))
    end_time = 3600.0 if max_t <= 3600.0 else 3600.0 + OT_LENGTH

    # ---- Build penalty "assessment batches" by exact timestamp ----
    # At each (t_call) we will assign coincidental pairs and create penalty objects.
    # We'll keep original Period/Time columns for debugging, but simulation uses t_call.
    batches = {}
    for t, g in pen_game.groupby("t_call", sort=True):
        batches[float(t)] = g.sort_values("Penalty_ID").to_dict("records")

    # ---- Goals by time (process AFTER penalties at same time) ----
    goals_by_time = defaultdict(list)
    for _, r in goals_game.sort_values("t_goal").iterrows():
        goals_by_time[float(r["t_goal"])].append(r.to_dict())

    # Master event times (pen calls, goals, plus end_time)
    times = sorted(set(list(batches.keys()) + list(goals_by_time.keys()) + [end_time, 0.0]))

    # Active serving penalties per team (these reduce skaters and their clocks are running)
    # Each item is dict with: pid, team, len_sec, remaining, is_major, is_coincidental, cancelable, start_t
    active = {away: [], home: []}
    # Waiting queue per team (penalties assessed but not yet serving due to max-2 rule)
    waiting = {away: deque(), home: deque()}

    # Ledger rows for output
    ledger = {}  # pid -> dict with call, start, end, canceled, flags
    for rec in pen_game.to_dict("records"):
        pid = int(rec["Penalty_ID"])
        ledger[pid] = {
            "Game_ID": game_id,
            "Penalty_ID": pid,
            "Team": str(rec["Team"]),
            "Period": rec.get("Period"),
            "Time": rec.get("Time"),
            "t_call": float(rec["t_call"]),
            "Pen_Length_Min": int(rec["Pen_Length"]),
            "double_half": int(rec.get("double_half", 0)),
            "is_major": int(rec["Pen_Length"]) == 5,
            "is_coincidental": False,  # set at assessment time
            "cancelable": int(rec["Pen_Length"]) == 2,  # minors only
            "serve_start": np.nan,
            "serve_end": np.nan,
            "ended_reason": None,       # "expired" | "canceled"
        }

    # Helper: start penalties from waiting if slots (<2) available
    def start_from_queue(team: str, now: float):
        while len(active[team]) < 2 and len(waiting[team]) > 0:
            p = waiting[team].popleft()
            p["start_t"] = now
            ledger[p["pid"]]["serve_start"] = now
            active[team].append(p)

    # Helper: compute skaters at time now based on active counts
    def compute_skaters(now: float):
        phase = "OT" if now >= 3600.0 else "REG"
        a_ct = len(active[away])
        h_ct = len(active[home])
        if phase == "REG":
            sA = max(3, 5 - a_ct)
            sH = max(3, 5 - h_ct)
            # With queueing, 3v3 in REG should be impossible; safety bump
            if sA == 3 and sH == 3:
                sA, sH = 4, 4
        else:
            # 3v3 baseline; penalties create 4v3 by adding skater to non-penalized side
            sA = 3 + max(0, h_ct - a_ct)
            sH = 3 + max(0, a_ct - h_ct)
        return int(sA), int(sH), phase

    # Helper: advance time and decrement active penalty clocks
    def advance(dt: float):
        if dt <= 0:
            return
        for team in [away, home]:
            for p in active[team]:
                p["remaining"] = max(0.0, p["remaining"] - dt)

    # Helper: expire penalties whose clocks hit 0
    def expire(now: float):
        changed = False
        for team in [away, home]:
            keep = []
            for p in active[team]:
                if p["remaining"] <= 1e-9:
                    pid = p["pid"]
                    ledger[pid]["serve_end"] = now
                    ledger[pid]["ended_reason"] = ledger[pid]["ended_reason"] or "expired"
                    changed = True
                else:
                    keep.append(p)
            active[team] = keep
        return changed

    # Helper: cancel one defending minor (non-coincidental, cancelable) if possible
    def cancel_one(def_team: str, now: float):
        # pick earliest-ending by remaining time (smallest remaining)
        candidates = [p for p in active[def_team] if (p["cancelable"] and not p["is_coincidental"] and not p["is_major"])]
        if not candidates:
            return False
        p = sorted(candidates, key=lambda x: x["remaining"])[0]
        pid = p["pid"]
        # remove it
        active[def_team] = [x for x in active[def_team] if x["pid"] != pid]
        ledger[pid]["serve_end"] = now
        ledger[pid]["ended_reason"] = "canceled"
        return True

    # Timeline segments
    timeline_rows = []
    cur = 0.0
    last_state = None  # (sA, sH, phase)

    # We'll iterate over event times but also need to jump to next expiration between events
    # So we do a while loop until end_time.
    def next_expiration_time():
        ends = []
        for team in [away, home]:
            for p in active[team]:
                ends.append(cur + p["remaining"])
        return min(ends) if ends else np.inf

    # For mismatch reporting and goal enrichment
    goal_rows = []
    mismatch_rows = []

    while cur < end_time - 1e-9:
        # next scheduled event time (pen calls or goals or end_time)
        future_times = [t for t in times if t > cur + 1e-9]
        next_event_t = min(future_times) if future_times else end_time

        # next expiration time
        next_exp_t = next_expiration_time()

        nxt = min(next_event_t, next_exp_t, end_time)

        # Create/extend timeline segment for [cur, nxt) using current state
        sA, sH, phase = compute_skaters(cur + 1e-6)
        state = (sA, sH, phase)
        if last_state is None:
            last_state = state
        # add segment
        if nxt > cur:
            timeline_rows.append({
                "Game_ID": game_id,
                "Start_t": cur,
                "End_t": nxt,
                "Phase": phase,
                "Away_Team": away,
                "Home_Team": home,
                "Away_Skaters": sA,
                "Home_Skaters": sH
            })

        # advance clocks
        advance(nxt - cur)
        cur = nxt

        # process expirations at cur
        expired = expire(cur)
        if expired:
            # starting queued penalties may become possible
            start_from_queue(away, cur)
            start_from_queue(home, cur)

        # ------------------------------------------------------------
        # IMPORTANT ORDERING RULE:
        # GOAL happens first, penalties assessed at same timestamp
        # apply AFTER the goal (they affect the next faceoff/play).
        # ------------------------------------------------------------

        # 1) process goals at this exact time using CURRENT manpower (pre-penalties)
        if cur in goals_by_time:
            for gr in goals_by_time[cur]:
                scoring = str(gr["Team"])
                defending = home if scoring == away else away

                # PRE-GOAL state (do NOT include penalties assessed at cur)
                sA, sH, phase = compute_skaters(cur + 1e-6)
                team_s = sA if scoring == away else sH
                opp_s  = sH if scoring == away else sA
                pre_sit = sit_label(team_s, opp_s, phase)

                # record goal w/ pre-goal state
                out = dict(gr)
                out.update({
                    "Away_Team": away,
                    "Home_Team": home,
                    "Computed_Phase": phase,
                    "PreGoal_Team_Skaters": int(team_s),
                    "PreGoal_Opp_Skaters": int(opp_s),
                    "PreGoal_Situation": pre_sit,
                })
                goal_rows.append(out)

                # mismatch check (same as before)...

                # cancellation rules (same as before):
                # ONLY cancel if strict advantage exists pre-goal
                if team_s > opp_s:
                    canceled = cancel_one(defending, cur)
                    if canceled:
                        start_from_queue(away, cur)
                        start_from_queue(home, cur)

        # 2) now apply penalties assessed at this same timestamp (post-goal stoppage)
        if cur in batches:
            batch = batches[cur]

            # coincidental pairing within this batch (same as before)
            by_team = defaultdict(list)
            for rec in batch:
                tm = str(rec["Team"])
                if tm == away or tm == home:
                    by_team[tm].append(int(rec["Penalty_ID"]))

            k = 0
            if (away in by_team) and (home in by_team):
                k = min(len(by_team[away]), len(by_team[home]))
            paired_away = set(by_team[away][:k]) if k > 0 else set()
            paired_home = set(by_team[home][:k]) if k > 0 else set()

            # enqueue penalties (they start after the goal; but since we're now post-goal,
            # starting them at cur is correct for subsequent play)
            for rec in batch:
                pid = int(rec["Penalty_ID"])
                team = str(rec["Team"])
                if team not in [away, home]:
                    continue

                Lmin = int(rec["Pen_Length"])
                obj = {
                    "pid": pid,
                    "team": team,
                    "len_sec": float(Lmin) * 60.0,
                    "remaining": float(Lmin) * 60.0,
                    "is_major": (Lmin == 5),
                    "is_coincidental": (pid in paired_away) or (pid in paired_home),
                    "cancelable": (Lmin == 2),
                    "start_t": None,
                }
                ledger[pid]["is_coincidental"] = obj["is_coincidental"]

                waiting[team].append(obj)

            start_from_queue(away, cur)
            start_from_queue(home, cur)

        


    # finalize ledger df
    led_df = pd.DataFrame(list(ledger.values()))
    tl_df = pd.DataFrame(timeline_rows)
    goals_df = pd.DataFrame(goal_rows)
    mism_df = pd.DataFrame(mismatch_rows)

    return led_df, tl_df, goals_df, mism_df

# ============================================================
# RUN ALL GAMES
# ============================================================
all_led = []
all_tl = []
all_go = []
all_mm = []

for _, gt in game_teams.iterrows():
    game_id = str(gt["Game_ID"])
    away = str(gt["Away_Team"])
    home = str(gt["Home_Team"])
    led_df, tl_df, goals_df, mism_df = simulate_game(game_id, away, home)
    all_led.append(led_df)
    all_tl.append(tl_df)
    all_go.append(goals_df)
    all_mm.append(mism_df)

penalty_ledger = pd.concat(all_led, ignore_index=True) if all_led else pd.DataFrame()
manpower_timeline = pd.concat(all_tl, ignore_index=True) if all_tl else pd.DataFrame()
goals_with_state = pd.concat(all_go, ignore_index=True) if all_go else pd.DataFrame()
mismatches = pd.concat(all_mm, ignore_index=True) if all_mm else pd.DataFrame()

# Save audit outputs
penalty_ledger.to_csv(temp_folder / "penalty_ledger.csv", index=False)
manpower_timeline.to_csv(temp_folder / "manpower_timeline.csv", index=False)
goals_with_state.to_csv(temp_folder / "goals_with_computed_manpower.csv", index=False)
mismatches.to_csv(temp_folder / "manpower_vs_ppnote_mismatches.csv", index=False)

print("Saved audit files:")
print(" -", temp_folder / "penalty_ledger.csv")
print(" -", temp_folder / "manpower_timeline.csv")
print(" -", temp_folder / "goals_with_computed_manpower.csv")
print(" -", temp_folder / "manpower_vs_ppnote_mismatches.csv")
print("Mismatch rows:", len(mismatches))

# ============================================================
# BUILD YOUR SUMMARY TABLES (long + wide)
# Situations requested + 4 stats: GF, GA, Instances, Time_Seconds
# ============================================================
requested = [
    "5v5","5v4","5v3","4v4","4v3",
    "3v3 (NON OT)","3v3 (OT)",
    "3v4","3v5","4v5"
]

# Time per Team×Situation from timeline
time_rows = []
for _, r in manpower_timeline.iterrows():
    dt = float(r["End_t"] - r["Start_t"])
    if dt <= 0:
        continue
    phase = r["Phase"]
    away = r["Away_Team"]
    home = r["Home_Team"]
    lab_away = sit_label(int(r["Away_Skaters"]), int(r["Home_Skaters"]), phase)
    lab_home = sit_label(int(r["Home_Skaters"]), int(r["Away_Skaters"]), phase)
    if lab_away in requested:
        time_rows.append({"Team": away, "Situation": lab_away, "Time_Seconds": dt})
    if lab_home in requested:
        time_rows.append({"Team": home, "Situation": lab_home, "Time_Seconds": dt})

time_df = pd.DataFrame(time_rows)
time_agg = (time_df.groupby(["Team","Situation"], as_index=False)["Time_Seconds"].sum()
            if not time_df.empty else pd.DataFrame(columns=["Team","Situation","Time_Seconds"]))

# Goals For by pre-goal situation
gf_df = goals_with_state[goals_with_state["PreGoal_Situation"].isin(requested)].copy()
gf_df["Goals_For"] = 1
gf_agg = gf_df.groupby(["Team","PreGoal_Situation"], as_index=False)["Goals_For"].sum()
gf_agg = gf_agg.rename(columns={"PreGoal_Situation":"Situation"})

# Goals Against: invert skaters at pre-goal
ga_rows = []
for _, r in goals_with_state.iterrows():
    sit = r["PreGoal_Situation"]
    if sit not in requested:
        continue
    scoring = str(r["Team"])
    away = str(r["Away_Team"])
    home = str(r["Home_Team"])
    defending = home if scoring == away else away
    phase = str(r["Computed_Phase"])
    ts = int(r["PreGoal_Team_Skaters"])
    os = int(r["PreGoal_Opp_Skaters"])
    def_sit = sit_label(os, ts, phase)
    if def_sit in requested:
        ga_rows.append({"Team": defending, "Situation": def_sit, "Goals_Against": 1})

ga_df = pd.DataFrame(ga_rows)
ga_agg = (ga_df.groupby(["Team","Situation"], as_index=False)["Goals_Against"].sum()
          if not ga_df.empty else pd.DataFrame(columns=["Team","Situation","Goals_Against"]))

# Instances: count transitions per game/team
inst_rows = []
for game_id, tg in manpower_timeline.groupby("Game_ID"):
    tg = tg.sort_values("Start_t")
    away = tg["Away_Team"].iloc[0]
    home = tg["Home_Team"].iloc[0]
    last = {away: None, home: None}
    for _, r in tg.iterrows():
        phase = r["Phase"]
        lab_away = sit_label(int(r["Away_Skaters"]), int(r["Home_Skaters"]), phase)
        lab_home = sit_label(int(r["Home_Skaters"]), int(r["Away_Skaters"]), phase)
        for team, lab in [(away, lab_away), (home, lab_home)]:
            if lab not in requested:
                continue
            if last[team] != lab:
                inst_rows.append({"Team": team, "Situation": lab, "Instances": 1})
                last[team] = lab

inst_df = pd.DataFrame(inst_rows)
inst_agg = (inst_df.groupby(["Team","Situation"], as_index=False)["Instances"].sum()
            if not inst_df.empty else pd.DataFrame(columns=["Team","Situation","Instances"]))

# Base grid
teams = sorted(set(manpower_timeline["Away_Team"].unique()).union(set(manpower_timeline["Home_Team"].unique())))
base = pd.MultiIndex.from_product([teams, requested], names=["Team","Situation"]).to_frame(index=False)

long_out = (base
            .merge(time_agg, on=["Team","Situation"], how="left")
            .merge(gf_agg,   on=["Team","Situation"], how="left")
            .merge(ga_agg,   on=["Team","Situation"], how="left")
            .merge(inst_agg, on=["Team","Situation"], how="left"))

for c in ["Time_Seconds","Goals_For","Goals_Against","Instances"]:
    long_out[c] = long_out[c].fillna(0)

long_out["Time_Seconds"] = long_out["Time_Seconds"].round(0).astype(int)
long_out["Goals_For"] = long_out["Goals_For"].astype(int)
long_out["Goals_Against"] = long_out["Goals_Against"].astype(int)
long_out["Instances"] = long_out["Instances"].astype(int)

wide_out = long_out.pivot_table(index="Team",
                                columns="Situation",
                                values=["Goals_For","Goals_Against","Instances","Time_Seconds"],
                                aggfunc="sum",
                                fill_value=0)
wide_out.columns = [f"{stat}__{sit}" for stat, sit in wide_out.columns]
wide_out = wide_out.reset_index()

long_out.to_csv(temp_folder / "pppk_situations_long.csv", index=False)
wide_out.to_csv(temp_folder / "pppk_situations_wide.csv", index=False)

print("Saved summary tables:")
print(" -", temp_folder / "pppk_situations_long.csv")
print(" -", temp_folder / "pppk_situations_wide.csv")


Saved audit files:
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\penalty_ledger.csv
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\manpower_timeline.csv
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\goals_with_computed_manpower.csv
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\manpower_vs_ppnote_mismatches.csv
Mismatch rows: 0
Saved summary tables:
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\pppk_situations_long.csv
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\pppk_situations_wide.csv


In [9]:
import pandas as pd
import numpy as np

tl = manpower_timeline.copy()
g = goals_with_state.copy()

game_tl = {gid: df.reset_index(drop=True) for gid, df in tl.groupby("Game_ID", sort=False)}

EPS = 1e-3  # 1 millisecond: "just before the goal"

def find_seg_idx(tg: pd.DataFrame, t: float):
    hit = tg.index[(tg["Start_t"] <= t) & (t < tg["End_t"])]
    return None if len(hit) == 0 else int(hit[0])

def run_start_time(tg: pd.DataFrame, team_side: str, seg_idx: int, run_type: str) -> float:
    """
    Walk backward to the start of the current advantaged/disadvantaged run.
    run_type: "ADV" or "DIS"
    """
    if team_side == "Away":
        ts_col, os_col = "Away_Skaters", "Home_Skaters"
    else:
        ts_col, os_col = "Home_Skaters", "Away_Skaters"

    def in_run(i):
        ts = int(tg.loc[i, ts_col])
        os = int(tg.loc[i, os_col])
        return (ts > os) if run_type == "ADV" else (ts < os)

    i = seg_idx
    while i - 1 in tg.index and in_run(i - 1):
        i -= 1
    return float(tg.loc[i, "Start_t"])

pp_rows = []
pk_rows = []

for _, r in g.iterrows():
    game_id = str(r["Game_ID"])
    if game_id not in game_tl:
        continue

    tg = game_tl[game_id]
    t_goal = float(r["t_goal"]) if "t_goal" in r else float(r.get("Goal_t", np.nan))
    if not np.isfinite(t_goal):
        continue

    away = str(r["Away_Team"])
    home = str(r["Home_Team"])
    scoring_team = str(r["Team"])
    defending_team = home if scoring_team == away else away

    # --- PRE-GOAL skaters: trust the pregoal fields (already fixed in simulator) ---
    team_s = int(r["PreGoal_Team_Skaters"])
    opp_s  = int(r["PreGoal_Opp_Skaters"])
    phase  = str(r["Computed_Phase"])

    # --- Timeline lookup must be "just before the goal" to avoid post-goal penalty segment ---
    t_query = max(0.0, t_goal - EPS)
    seg_idx = find_seg_idx(tg, t_query)
    if seg_idx is None:
        continue

    scoring_side = "Away" if scoring_team == away else "Home"
    defending_side = "Away" if defending_team == away else "Home"

    # PP goals scored: scoring team had advantage PRE-goal
    if team_s > opp_s:
        adv_start = run_start_time(tg, scoring_side, seg_idx, "ADV")
        pp_rows.append({
            "Game_ID": game_id,
            "Team": scoring_team,
            "Opponent": defending_team,
            "Period": r.get("Period", None),
            "Time": r.get("Time", None),
            "Goal_t": t_goal,
            "Phase": phase,
            "Team_Skaters": team_s,
            "Opp_Skaters": opp_s,
            "Advantage_Start_t": adv_start,
            "Delta_Seconds": t_goal - adv_start,
            "PP_Note": r.get("PP", np.nan),
        })

    # PK goals allowed: defending team was disadvantaged PRE-goal
    if opp_s > team_s:  # i.e., scoring had advantage => defending short
        dis_start = run_start_time(tg, defending_side, seg_idx, "DIS")
        pk_rows.append({
            "Game_ID": game_id,
            "Team": defending_team,
            "Opponent": scoring_team,
            "Period": r.get("Period", None),
            "Time": r.get("Time", None),
            "Goal_t": t_goal,
            "Phase": phase,
            "Team_Skaters": opp_s,   # defending skaters are "opp" from scorer perspective
            "Opp_Skaters": team_s,   # scoring skaters
            "Disadvantage_Start_t": dis_start,
            "Delta_Seconds": t_goal - dis_start,
            "PP_Note": r.get("PP", np.nan),
        })

pp_df = pd.DataFrame(pp_rows).sort_values(["Team","Delta_Seconds","Game_ID","Goal_t"]).reset_index(drop=True)
pk_df = pd.DataFrame(pk_rows).sort_values(["Team","Delta_Seconds","Game_ID","Goal_t"]).reset_index(drop=True)

events = pd.concat([
    pp_df.assign(Event_Type="PP_Goal_For"),
    pk_df.assign(Event_Type="PP_Goal_Against_While_SH"),
], ignore_index=True, sort=False).sort_values(
    ["Team","Event_Type","Delta_Seconds","Game_ID","Goal_t"]
).reset_index(drop=True)

pp_out = temp_folder / "pp_goal_time_to_score.csv"
pk_out = temp_folder / "pk_goal_time_to_allow.csv"
ev_out = temp_folder / "pp_pk_goal_timing_events.csv"

pp_df.to_csv(pp_out, index=False)
pk_df.to_csv(pk_out, index=False)
events.to_csv(ev_out, index=False)

print("Saved:")
print(" -", pp_out, "| rows:", len(pp_df))
print(" -", pk_out, "| rows:", len(pk_df))
print(" -", ev_out, "| rows:", len(events))


Saved:
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\pp_goal_time_to_score.csv | rows: 1102
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\pk_goal_time_to_allow.csv | rows: 214
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\PEN\pp_pk_goal_timing_events.csv | rows: 1316
